In [1]:
# imports

import os
import random
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

from google import genai
from google.genai import types

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [41]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [2]:
message = "Hello there, are you GPT? This is my first ever message to you! Hi!"

messages = [
    {"role": "system", 
    "content": "CRITICAL: You have a hard budget of 60 characters total. Output ONLY the clean answer string. Do not think. Do not explain."}, 
    {"role": "user", "content": message}
    ]

messages

[{'role': 'system',
  'content': 'CRITICAL: You have a hard budget of 60 characters total. Output ONLY the clean answer string. Do not think. Do not explain.'},
 {'role': 'user',
  'content': 'Hello there, are you GPT? This is my first ever message to you! Hi!'}]

In [32]:
GEMMA4_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"


openai = OpenAI()



In [37]:
gemma = OpenAI(base_url=GEMMA4_BASE_URL, api_key=api_key) #, temperature= 0.0, max_output_tokens= 20)
response = gemma.chat.completions.create(
    model="gemma-4-31b-it", messages= messages, temperature= 0.0, max_tokens=40,reasoning_effort="none")

# response.choices[0].message.content
display(Markdown(response.choices[0].message.content))

BadRequestError: Error code: 400 - [{'error': {'code': 400, 'message': 'Thinking budget is not supported for this model.', 'status': 'INVALID_ARGUMENT'}}]

In [2]:
def get_dialog(person, dialog_num, turn):
    """Simulate a dialog line for a person."""
    messages = {
        "A": [
            "Hey everyone, what are we doing this weekend?",
            "I was thinking we could go hiking.",
            "The weather forecast looks great for Saturday.",
            "Should we pack lunch or eat out?",
            "Let's meet at the trailhead at 9am then!",
        ],
        "B": [
            "Welcome! I'm so glad you're guys here.",
            "Hiking sounds awesome, I know a great trail.",
            "Perfect, I'll bring my camera then.",
            "Let's pack lunch, it's cheaper and more fun.",
            "Sounds like a plan, I'll bring extra water!",
        ],
        "C": [
            "Count me in! I've been bored all week.",
            "Oh I love hiking, which trail are you thinking?",
            "Great, I'll charge my phone for photos.",
            "I can make sandwiches for everyone if you want.",
            "Can't wait, this is going to be so fun!",
        ],
        "D": [
            "Count me in! ",
            "Oh I love hiking",
            "Great",
            "Who can make sandwiches?",
            "Can't wait!",
        ],
    }
    # Use dialog_num and turn to pick a somewhat varied line
    idx = (dialog_num + turn) % len(messages[person])
    return f"[Dialog {dialog_num+1}] Person {person}: {messages[person][idx]}"

def store_line(person, line):
    """Store a dialog line in the appropriate list."""
    if person == "A":
        listA.append(line)
    elif person == "B":
        listB.append(line)
    elif person == "C":
        listC.append(line)
    # elif person == "D":
    #     listC.append(line)

def run_conversation(num_dialogs=5):
    """
    Run a conversation of `num_dialogs` rounds.

    Each dialog round has 3 turns (one per person):
      - Dialog 1: random starter, then the other two in random order.
      - Dialog N (N > 1): the FIRST RESPONDER (2nd speaker) of the previous
        dialog becomes the new starter, then the remaining two in random order.
    """
    first_speaker = random.choice(persons)  # Random starter for dialog 1

    print("=" * 50)
    print("        THREE-PERSON CONVERSATION")
    print("=" * 50)

    for dialog_num in range(num_dialogs):
        # Build the ordered speaker list for this dialog
        others = [p for p in persons if p != first_speaker]
        random.shuffle(others)
        order = [first_speaker] + others  # first speaker is fixed; others shuffled

        print(f"\n--- Dialog {dialog_num + 1} ---")
        print(f"  Turn order: {' → '.join(order)}")

        for turn, speaker in enumerate(order):
            line = get_dialog(speaker, dialog_num, turn)
            store_line(speaker, line)
            print(f"  {line}")

        # Rule 3: next dialog's starter is the FIRST RESPONDER (2nd speaker) of this dialog
        first_speaker = order[1]

    print("\n" + "=" * 50)

# # Run the conversation
# run_conversation(5)

# # Display stored lists
# print("\n📋 STORED DIALOG LISTS")
# print("=" * 50)

# print("\n🗂  listA (Person A's lines):")
# for line in listA:
#     print(f"  {line}")

# print("\n🗂  listB (Person B's lines):")
# for line in listB:
#     print(f"  {line}")

# print("\n🗂  listC (Person C's lines):")
# for line in listC:
#     print(f"  {line}")

# print(f"\n✅ Totals → A: {len(listA)} | B: {len(listB)} | C: {len(listC)}")

In [11]:
# Storage lists for each person's dialogs
listA = []
listB = []
listC = []
# listD = []

persons = ["A", "B", "C"] # , "D"]
run_conversation(5)

        THREE-PERSON CONVERSATION

--- Dialog 1 ---
  Turn order: A → B → C
  [Dialog 1] Person A: Hey everyone, what are we doing this weekend?
  [Dialog 1] Person B: Hiking sounds awesome, I know a great trail.
  [Dialog 1] Person C: Great, I'll charge my phone for photos.

--- Dialog 2 ---
  Turn order: B → C → A
  [Dialog 2] Person B: Hiking sounds awesome, I know a great trail.
  [Dialog 2] Person C: Great, I'll charge my phone for photos.
  [Dialog 2] Person A: Should we pack lunch or eat out?

--- Dialog 3 ---
  Turn order: C → B → A
  [Dialog 3] Person C: Great, I'll charge my phone for photos.
  [Dialog 3] Person B: Let's pack lunch, it's cheaper and more fun.
  [Dialog 3] Person A: Let's meet at the trailhead at 9am then!

--- Dialog 4 ---
  Turn order: B → A → C
  [Dialog 4] Person B: Let's pack lunch, it's cheaper and more fun.
  [Dialog 4] Person A: Let's meet at the trailhead at 9am then!
  [Dialog 4] Person C: Count me in! I've been bored all week.

--- Dialog 5 ---
  Tu

In [35]:
response.__dict__

{'id': 'sacjapHGAbOGz7IP74vB4AU',
 'choices': [Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='<thought>*   User: "Hello there, are you GPT?</thought>', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought': True}}))],
 'created': 1780721585,
 'model': 'gemma-4-31b-it',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': CompletionUsage(completion_tokens=0, prompt_tokens=50, total_tokens=62, completion_tokens_details=None, prompt_tokens_details=None),
 '_request_id': None}

In [5]:


# Initialize the native Google client
# It automatically picks up the GEMINI_API_KEY environment variable
client = genai.Client()

# 1. Store only user/assistant flow in the contents array
messages = [
   types.Content(
        role="user",
        parts=[types.Part.from_text(text="Hello there, are you GPT? This is my first ever message to you! Hi!")]
    )
]

# 2. Put system rules, zero thinking budget, and constraints here
config = types.GenerateContentConfig(
    system_instruction="Answer in under 60 characters. Do not explain. Be brief.",
    # temperature=0.0,
    # max_output_tokens=30, # Low ceiling to prevent rambling
    # thinking_config=types.ThinkingConfig(
        # thinking_budget=0  # Disables the internal reasoning channel entirely
    # )
)

# 3. Call the model using the native client
response = client.models.generate_content(
    model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
    contents=messages,
    config=config
)

print(response.text)


Hi! I am a large language model trained by Google.


In [6]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""*   User says: "Hello there, are you GPT? This is my first ever message to you! Hi!"
    *   Constraint 1: Under 60 characters.
    *   Constraint 2: Do not explain.
    *   Constraint 3: Be brief.

    *   I am a large language model trained by Google. (Wait, I need to check the system prompt or standard identity. Usually, if not specified, I identify as a model trained by Google).
    *   User asked if I am "GPT" (which refers to OpenAI's models).

    *   *Draft 1:* No, I am a large language model trained by Google. (48 chars)
    *   *Draft 2:* Hello! I am a model trained by Google. (36 chars)
    *   *Draft 3:* Hi! I'm a large language model trained by Google. (47 chars)

    *   Under 60 chars? Yes.
    *   No explanation? Yes.
    *   Brief? Yes.

    *   "Hi! I am a large language model trained by Google." (49